# Day 2 Lab — Build a Multi-Agent Research Team

Yesterday you built a **single agent**: one workflow, function nodes, and a router *you* wrote deciding the flow. Today you build a **multi-agent system**: a team of specialized LLM agents coordinated by an **LLM supervisor** that decides — at runtime — who works next.

| | Day 1 — single agent | Day 2 — multi-agent |
|---|---|---|
| Nodes | Python functions | LLM agents with personas |
| Routing | your `if/else` (`quality_router`) | a supervisor **LLM** decides at runtime |
| Prompts | one prompt for everything | one system prompt **per agent** |
| Tools | available everywhere | **scoped** — only the researcher can search |
| Loop | quality-score retry | critic sends the draft back to the writer |

**What does NOT change:** State + Nodes + Edges. A multi-agent system is *still* just a `StateGraph`. The new ideas are personas, the supervisor, and guardrails.

```
             ┌──────────── supervisor ─────────────┐
             │       (LLM decides who's next)      │
    ┌────────┼───────────┬───────────┬─────────────┤
    ↓        ↓           ↓           ↓             ↓
 researcher  analyst    writer     critic       FINISH
    │        │           │           │             ↓
    └────────┴───────────┴───────────┘            END
         (every worker reports back to the supervisor)
```

**Read before you start (~25 min):**
1. [Multi-agent concepts](https://docs.langchain.com/oss/python/langgraph/multi-agent) — architectures, supervisor pattern
2. [Conditional branching + loops](https://docs.langchain.com/oss/python/langgraph/use-graph-api#conditional-branching) — refresher, you need both again
3. [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output) — the supervisor's decision is structured!

**Setup:** same as Day 1 — `uv sync`, keys in `.env`, or `USE_FAKE=1`.

In [ ]:
import os
import operator
from datetime import datetime
from typing import Annotated, List, Literal
from typing_extensions import TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, SystemMessage

# TODO STEP 0 — same imports as Day 1:
# StateGraph, START, END from langgraph.graph
# InMemorySaver from langgraph.checkpoint.memory

load_dotenv()

MAX_REVISIONS = 2      # cap on writer↔critic loops
MAX_TURNS = 12         # cap on total supervisor decisions

## STEP 1 — SHARED STATE: the team's "blackboard"
Day 1's state was a data PIPELINE (each field filled once, in
order). Day 2's state is a BLACKBOARD: every agent reads all of
it and writes only its own section; the supervisor reads it to
decide who goes next.

Define a TypedDict with:
  task (str)
  research_notes  <- List[str], APPEND-ONLY (which reducer? Day 1!)
  analysis (str), draft (str), critique (str)
  revision_count (int), turn_count (int)
  next_agent (str)   <- the supervisor writes its decision HERE
  execution_logs     <- append-only, same as Day 1

ASK YOURSELF: why must research_notes append but draft overwrite?
What would happen to the revision loop if draft used operator.add?

In [ ]:
class TeamState(TypedDict):
    task: str
    # TODO: add the remaining 8 keys (two use Annotated + operator.add)
    pass

## STEP 2 — STRUCTURED ROUTING DECISION
Day 1: structured output produced a quality SCORE.
Day 2: structured output produces a ROUTING DECISION — this is
the trick that turns an LLM into a supervisor. Literal[...] means
the model CANNOT invent an agent that doesn't exist.

WHERE TO LOOK: structured-output docs (same page as Day 1).

In [ ]:
class RouterDecision(BaseModel):
    """The supervisor's choice of who acts next."""
    next_agent: Literal["researcher", "analyst", "writer", "critic", "FINISH"]
    reason: str = Field(description="One sentence explaining the choice")

## STEP 3 — ONE LLM, FOUR PERSONAS (+ tools scoped per agent)
A multi-agent "team" doesn't need four models — it needs four
SYSTEM PROMPTS. (In production you might also vary the model per
agent: cheap model for the critic, big one for the writer.)

TODO:
1. Write a PERSONAS dict: role -> system prompt, for
   "researcher", "analyst", "writer", "critic".
   Each persona must say what the agent DOES and what it MUST NOT
   do (e.g. the researcher never analyzes). Boundaries between
   agents live in the prompts — write them sharp.
2. Create llm (ChatOpenAI + OpenRouter, exactly like Day 1) and
   search_tool (TavilySearch(max_results=4)).
3. supervisor_llm = llm.with_structured_output(RouterDecision)
4. Helper: run_persona(role, user_content) → invoke llm with
   [SystemMessage(PERSONAS[role]), HumanMessage(user_content)]
   and return response.content.

TOOL SCOPING: only the researcher node may call search_tool.
That's a deliberate design decision, not a limitation — ask
yourself what could go wrong if the critic could search.

In [ ]:
PERSONAS = {
    # TODO: four personas
}

# TODO: llm, search_tool, supervisor_llm, run_persona

## STEP 4 — THE SUPERVISOR NODE (the piece Day 1 didn't have)
The supervisor node must:
1. Increment turn_count.
2. Build a STATUS SUMMARY of the blackboard (which sections are
   filled? what does the critique say? how many revisions?).
   Don't dump the full text of everything — the supervisor needs
   STATUS, not content. (Why? Think tokens and attention.)
3. Ask supervisor_llm for a RouterDecision.
4. GUARDRAILS — never trust an LLM to terminate a loop:
     a) if turn_count > MAX_TURNS → force FINISH
     b) if the LLM picks writer/critic but revision_count >=
        MAX_REVISIONS and a draft exists → force FINISH
   This is Day 1's iteration cap wearing a new hat. Same lesson:
   the LLM proposes, YOUR CODE disposes.
5. Return {"next_agent": ..., "turn_count": ..., "execution_logs": [...]}

WHERE TO LOOK: multi-agent docs → "Supervisor" section.

In [ ]:
def supervisor_node(state: TeamState):
    # TODO
    pass

## STEP 5 — WORKER AGENT NODES
Each worker: read the blackboard → act in persona → return a
PARTIAL update with ONLY its own section (Day 1 rule, unchanged).

In [ ]:
def researcher_node(state: TeamState):
    """Search the web (ONLY this agent may), condense to notes."""
    # TODO:
    # 1. results = search_tool.invoke({"query": state["task"]})["results"]
    # 2. Format results into a raw text block (title, content, url)
    # 3. notes = run_persona("researcher", f"Task ...\n\nSearch results:\n{raw}")
    # 4. return {"research_notes": [notes], "execution_logs": [...]}
    #    ^ note the LIST — research_notes is append-only!
    pass


def analyst_node(state: TeamState):
    """Turn raw notes into analysis."""
    # TODO: run_persona("analyst", ...) → {"analysis": ..., "execution_logs": [...]}
    pass


def writer_node(state: TeamState):
    """Write the draft — or REVISE it if a critique is present."""
    # TODO:
    # 1. revising = critique exists and starts with "REVISE"
    # 2. Build the prompt; when revising, include the previous draft
    #    AND the critique so the writer knows what to fix.
    # 3. return {"draft": ...,
    #            "critique": "",   <- WHY reset this? (see self-check)
    #            "revision_count": +1 only when revising,
    #            "execution_logs": [...]}
    pass


def critic_node(state: TeamState):
    """Review the draft against the research notes."""
    # TODO: run_persona("critic", ...) → the persona replies either
    # "APPROVED" or "REVISE: <fixes>". Store it in critique.
    pass

## STEP 6 — ROUTING FUNCTION + WIRE THE GRAPH
The conditional-edge function is now TRIVIAL — it just reads the
supervisor's decision:

    def route_from_supervisor(state) -> str:
        return state["next_agent"]

Compare with Day 1, where all decision logic lived inside
quality_router. The intelligence MOVED from the edge into a node.

Wiring checklist:
1. add all five nodes
2. START → supervisor
3. add_conditional_edges("supervisor", route_from_supervisor,
       {"researcher": "researcher", "analyst": "analyst",
        "writer": "writer", "critic": "critic", "FINISH": END})
4. EVERY worker gets an edge BACK to supervisor — the
   hub-and-spoke shape that defines the supervisor pattern.
   (A for-loop over the four worker names is idiomatic.)

In [ ]:
# TODO: route_from_supervisor + graph wiring

## STEP 7 — COMPILE, VISUALIZE, RUN
Same as Day 1: compile with InMemorySaver, print the Mermaid
diagram (it should look like a STAR, not Day 1's chain), stream
with stream_mode="values" and a thread_id, print the final draft.

EXPERIMENT 1: set MAX_REVISIONS = 0. What happens to quality?
EXPERIMENT 2: delete guardrail (a) and make the critic always
  say REVISE. Watch the turn cap save you — then delete guardrail
  (b) too and meet your old friend GraphRecursionError.
EXPERIMENT 3: swap the analyst's persona for a terrible one
  ("you are vague and generic"). How far does the damage spread
  through the team? This is why persona boundaries matter.

In [ ]:
initial_state = {
    "task": "Should our company adopt multi-agent AI systems in 2026?",
    "research_notes": [],
    "analysis": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "turn_count": 0,
    "next_agent": "",
    "execution_logs": [],
}
# TODO: compile, visualize, stream, print final draft + stats

## SELF-CHECK before you look at the solution
[ ] I can explain the supervisor pattern in one sentence
[ ] My routing function reads state — the DECISION was made in a node
[ ] research_notes appends; draft overwrites; I know why each
[ ] The writer RESETS critique — I can explain what breaks if not
    (hint: what does the supervisor see on the turn after a revision?)
[ ] Only researcher_node touches search_tool
[ ] My supervisor has BOTH guardrails, and I triggered EXPERIMENT 2
[ ] My Mermaid diagram is a star: supervisor in the middle
[ ] I can name one task where Day 1's single agent is the BETTER
    design (multi-agent is not free: more calls, more latency,
    more places to break — coordination must earn its cost)

Stuck? Debugging order that works:
  1. stream_mode="updates" — watch each supervisor decision + reason
  2. print the status summary your supervisor_node builds — is the
     LLM seeing an accurate picture of the blackboard?
  3. check your conditional-edge dict covers ALL five decisions
  4. only THEN open day2_lab_solution.py